In [ ]:
!pip install qiskit
!pip install qiskit-ibm-runtime
!pip install qiskit[visualization]
!pip install qiskit-aer

In [ ]:
import numpy as np
from numpy import pi
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector
from collections import defaultdict
from qiskit.circuit.library import QFT
import matplotlib.pyplot as plt

## 0. Preliminaries — 결과 읽는 방법

이 노트북에서는 `q_simulation` 함수로 회로를 실행하고 측정 결과를 확인합니다.

먼저 알아둘 점이 하나 있습니다. **Qiskit은 큐비트를 오른쪽부터 왼쪽으로 셉니다.** 즉 0번 큐비트의 값이 결과 문자열의 *가장 오른쪽*에 놓입니다 (little-endian). 하지만 우리는 보통 숫자를 읽듯이 0번 큐비트를 *가장 왼쪽*에서 읽는 게 편합니다 (big-endian).

그래서 `q_simulation`은 모든 결과 문자열을 그냥 **뒤집어서** (`i[::-1]`) 0번 큐비트가 왼쪽에 오도록 맞춰줍니다. 그런 다음 각 결과가 몇 번 나왔는지 히스토그램으로 그려줍니다.

`shots`는 회로를 몇 번 실행할지를 정하고 (많을수록 확률 분포가 매끄러워집니다), `return_count=True`로 두면 그림 대신 결과 횟수(counts)를 그대로 돌려줍니다.

In [ ]:
def q_simulation(qc,shots=1000,return_count=False) :
    simulator = Aer.get_backend('aer_simulator')

    # 회로 컴파일
    compiled_circuit = transpile(qc, simulator)

    # 실행
    job = simulator.run(compiled_circuit, shots=shots)
    result = job.result()

    # 결과 확인
    counts = result.get_counts()

    # 시각화 위해서 앞 뒤 순서 바꾸기
    new_keys = []
    for i in counts.keys():
        new_keys.append( i[::-1] )

    counts_reversed = dict( zip(new_keys,counts.values()) )

    if return_count == True :
        return counts_reversed
    # 히스토그램 시각화
    return plot_histogram(counts_reversed)

# 1. Deutsch's algorithm

## 핵심 아이디어: 함수에게 몇 번이나 "물어보는가"

이 문제들에서 입력은 숨겨진 함수 $f$ 이고, 이 함수는 **oracle**이라는 검은 상자 안에 들어 있습니다. 우리는 안을 들여다볼 수 없고, 값을 하나 넣고 나오는 값을 읽을 수만 있습니다. 이렇게 한 번 물어보는 것을 **query**라고 부르며, 가능한 한 적게 묻는 것이 목표입니다.

바로 여기서 양자 컴퓨터가 강력해집니다. 이 문제들에서는 양자 컴퓨터가 고전 컴퓨터보다 **훨씬 적은 query**로 답을 알아낼 수 있습니다. 이 차이가 우리가 보여주려는 "quantum advantage"입니다.

## Deutsch의 문제

1비트를 받아 1비트를 내보내는 함수 $f$ 가 주어집니다. 이 함수는 둘 중 하나라고 약속(promise)되어 있습니다.

- **constant (상수)**: 항상 같은 값을 출력 ($f(0) = f(1)$), 또는
- **balanced (균형)**: 한 입력엔 $0$, 다른 입력엔 $1$ 을 출력 ($f(0) \neq f(1)$).

**질문:** 이 함수는 constant인가, balanced인가?

고전 컴퓨터는 확실히 하려면 입력 두 개를 모두 확인해야 하므로 **query가 2번** 필요합니다. Deutsch의 알고리즘은 단 **1번의 query** 로 답합니다.

### 오라클(4가지 함수)을 만드는 양자 회로

1비트 → 1비트 함수는 딱 4가지뿐이라, 만들 오라클도 4개입니다. 오라클은 다음처럼 동작하는 2-큐비트 게이트입니다.

$$
|x\rangle|y\rangle \;\longrightarrow\; |x\rangle|y \oplus f(x)\rangle
$$

여기서 $x$ 는 입력 큐비트, $y$ 는 보조(출력) 큐비트, $\oplus$ 는 XOR입니다 ($1\oplus1=0$). $y$ 에 그냥 덮어쓰지 않고 XOR을 쓰는 이유는, 모든 양자 게이트가 되돌릴 수 있어야(reversible) 하기 때문입니다.

| case | 함수 | 종류 |
|------|----------|------|
| 1 | $f(x)=0$ | constant |
| 2 | $f(x)=x$ | balanced |
| 3 | $f(x)=\text{NOT }x$ | balanced |
| 4 | $f(x)=1$ | constant |

이 4가지는 간단한 두 게이트만으로 전부 만들 수 있습니다.

- **CNOT** (`cx(0,1)`)은 입력을 보조 큐비트에 더해줍니다 → $x$ 에 따라 값이 바뀌는 부분을 담당
- **X** 게이트 (`x(1)`)는 보조 큐비트를 뒤집습니다 → 상수를 더하는 부분을 담당

그래서 코드가 case 2, 3에는 `cx(0,1)`을, case 3, 4에는 `x(1)`을 적용하는 것입니다.

오라클의 동작: $|x\rangle|y\rangle \rightarrow |x\rangle|y \oplus f(x)\rangle$

In [ ]:
def deutsch_function(case: int):
    # This function generates a quantum circuit for one of the 4 functions
    # from one bit to one bit

    if case not in [1, 2, 3, 4]:
        raise ValueError("`case` must be 1, 2, 3, or 4.")

    f = QuantumCircuit(2)
    if case in [2, 3]:
        f.cx(0, 1)
    if case in [3, 4]:
        f.x(1)
    return f

In [ ]:
# constant Zero f(x)=0
display(deutsch_function(1).draw(output="mpl"))

In [ ]:
# Balanced Identity f(x)=x
display(deutsch_function(2).draw(output="mpl"))

In [ ]:
# Balanced NOT f(x)= NOT x
display(deutsch_function(3).draw(output="mpl"))

In [ ]:
# Constant One f(x)=1
display(deutsch_function(4).draw(output="mpl"))

#### Note. Oracle이 진짜 constant, balanced인지 simulation으로 확인하는 방법
1. q0에 H를 걸어 0과 1을 동시에 입력으로 넣습니다(중첩).
2. 오라클이 f(0), f(1) 결과값을 계산하여 q1에 저장합니다.
3. q1을 측정합니다. <br>-constant이면 f(0)=f(1): 측정 결과는 항상 0 or 항상 1 <br> -balanaced면 0과 1이 50:50 확률로 섞여서 나옴

#### 핵심 트릭 — 함수의 답을 "부호"로 바꾸기

Deutsch의 알고리즘은 깔끔한 트릭 하나로 작동합니다. 보조 큐비트를 다음 특별한 상태로 만들어 두면,

$$
|-\rangle = \frac{1}{\sqrt{2}}\big(|0\rangle - |1\rangle\big)
$$

오라클을 통과시켜도 보조 큐비트는 전혀 변하지 않고, 대신 $f(x)$ 에 따라 입력 앞에 **플러스/마이너스 부호**가 붙습니다.

$$
|x\rangle|-\rangle \;\longrightarrow\; (-1)^{f(x)}\,|x\rangle|-\rangle
$$

즉 함수의 답이 입력 쪽으로 "부호"가 되어 되돌아옵니다(kickback). 여기에 Hadamard를 한 번 더 걸면 이 부호들이 서로 간섭하면서,

- $f$ 가 **constant**면 → 측정 결과 **0**
- $f$ 가 **balanced**면 → 측정 결과 **1**

단 한 번의 측정으로 답을 알 수 있습니다.

In [ ]:
# 결과 확인용 오라클 / 오라클 설계시 measure 는 포함하면 안됨
oracle_for_visual = QuantumCircuit(2,1)
oracle_for_visual.h(0)
oracle_for_visual.barrier()
oracle_for_visual.append(deutsch_function(4),[0,1])
oracle_for_visual.barrier()
oracle_for_visual.measure(1,0)
oracle_for_visual.draw("mpl")


In [ ]:
# Checking Constant / Balance
q_simulation(oracle_for_visual)

In [ ]:
# 결과 확인용 오라클 / 오라클 설계시 measure 는 포함하면 안됨
oracle_for_visual = QuantumCircuit(2,1)
oracle_for_visual.h(0)
oracle_for_visual.barrier()
oracle_for_visual.append(deutsch_function(3),[0,1])
oracle_for_visual.barrier()
oracle_for_visual.measure(1,0)
oracle_for_visual.draw("mpl")


In [ ]:
# Checking Constant / Balance
q_simulation(oracle_for_visual)

# Problem 1

Alice는 정체를 알 수 없는 함수 $f$ 가 들어 있는 오라클 하나를 건네받았습니다. 이 오라클은 `deutsch_function(3)`으로 구현되어 있고, **constant(상수, 입력과 무관하게 항상 같은 값)** 이거나 **balanced(균형, 입력의 절반은 0·절반은 1)** 둘 중 하나입니다.

고전 컴퓨터라면 정체를 밝히려고 함수를 **최소 2번** 호출해야 하지만, Alice는 양자 컴퓨터로 **단 한 번의 실행** 만에 이 오라클이 balanced임을 알아내려 합니다.

**요구사항**
- `deutsch_function(3)`을 오라클로 사용하여, 이 함수가 balanced임을 판별하는 양자 회로를 완성하세요.
- `q_simulation(qc, 1)`의 결과가 Deutsch 알고리즘의 이론적 결과와 일치해야 합니다. (balanced이므로 측정 결과는 **1**)

In [ ]:
# 코드를 완성하시오.
balanced_oracle = deutsch_function(3)
balanced_oracle.name = "Balanced_Oracle_3"
qc = QuantumCircuit(2, 1)

# Step 1: 초기화
       # 결과 큐비트 |1⟩
qc.barrier()
     # Hadamard 변환
qc.barrier()

# Step 2: 오라클 3 Balanced NOT
qc.append(balanced_oracle,[0,1])
qc.barrier()

# Step 3: 다시 Hadamard


# Step 4: 측정
qc.measure(0, 0)
qc.draw("mpl")

In [ ]:
q_simulation(qc)

# 2. Deutsch–Jozsa Algorithm

Deutsch–Jozsa는 Deutsch와 같은 아이디어인데, 이번엔 함수가 **$n$ 비트** 를 입력으로 받습니다 (여기선 $n=3$). 역시 함수는 둘 중 하나로 약속되어 있습니다.

- **constant**: 모든 입력에 같은 값, 또는
- **balanced**: 전체 입력의 정확히 절반은 $0$, 나머지 절반은 $1$.

**이득이 훨씬 커집니다.** 고전 컴퓨터는 확실히 하려면 전체 입력의 절반 넘게 확인해야 할 수도 있는데, 이는 $2^{n-1}+1$ 번의 query로 매우 빠르게 늘어납니다. 반면 양자 알고리즘은 $n$ 이 아무리 커도 여전히 **단 1번의 query** 면 됩니다.

순서는 Deutsch 알고리즘의 $n$-큐비트 버전입니다.

1. 입력 큐비트 $n$ 개는 $|0\rangle$, 보조 큐비트는 $|1\rangle$ 로 둡니다.
2. 모든 큐비트에 Hadamard를 겁니다.
3. 오라클을 한 번 통과시킵니다.
4. 입력 큐비트에 다시 Hadamard를 겁니다.
5. 입력 큐비트를 측정합니다.

결과 읽는 규칙은 간단합니다. **전부 0이면 constant, 하나라도 1이 있으면 balanced.**

### 더 큰 오라클 만들기 (Balanced)

알고리즘을 테스트하려면 예시 balanced 오라클이 필요합니다. 입력 큐비트 3개 ($q_0,q_1,q_2$)와 보조 큐비트 $q_3$ 를 쓰고, 보조 큐비트로 향하는 CNOT 3개를 적용합니다.

$$
\text{cx}(0,3),\ \text{cx}(1,3),\ \text{cx}(2,3)
$$

이것은 입력에 들어 있는 1의 개수가 홀수인지 짝수인지(**parity**, 패리티)를 계산합니다.

$$
f(x_0,x_1,x_2) = x_0 \oplus x_1 \oplus x_2
$$

패리티는 balanced입니다. 8개의 입력 중 정확히 절반이 $0$, 절반이 $1$ 을 줍니다. 아래 확인용 회로(입력에 Hadamard를 건 뒤 보조 큐비트 측정)를 돌리면 대략 $50{:}50$ 으로 갈리는 걸 볼 수 있고, 이로써 balanced임을 확인합니다.

In [ ]:
oracle = QuantumCircuit(4,name='Random Oracle')
oracle.cx(0,3)
oracle.cx(1,3)
oracle.cx(2,3)
oracle.draw("mpl")

In [ ]:
# 결과 확인용 오라클 / 오라클 설계시 measure 는 포함하면 안됨
oracle_for_visual = QuantumCircuit(4,1)
oracle_for_visual.h([0,1,2])

oracle_for_visual.barrier() # Oracle Starts
oracle_for_visual.cx(0,3)
oracle_for_visual.cx(1,3)
oracle_for_visual.cx(2,3)
oracle_for_visual.barrier()

oracle_for_visual.measure(3,0)
oracle_for_visual.draw("mpl")

In [ ]:
# Checking Constant / Balance
q_simulation(oracle_for_visual)

### Deutsch–Jozsa 알고리즘 구현하기

이제 전체를 입력 큐비트 3개와 보조 큐비트 1개로 조립합니다.

- **Step 1 — 준비:** `x(3)`으로 보조 큐비트를 $|1\rangle$ 로 만들고, `h([0,1,2,3])`으로 입력을 모든 경우에 퍼뜨립니다.
- **Step 2 — query:** 오라클을 한 번 추가합니다.
- **Step 3 — 합치기:** 입력 큐비트에 `h([0,1,2])`.
- **Step 4 — 측정:** 입력 큐비트를 측정합니다.

이 오라클은 balanced이므로 `000`은 **절대 나오지 않고**, 항상 다른 문자열이 나옵니다. 1 shot이면 충분합니다.

In [ ]:
qc = QuantumCircuit(4, 3)

# Step 1: 초기화
qc.x(3)           # 결과 큐비트 |1⟩
qc.barrier()
qc.h([0,1,2,3])      # Hadamard 변환
qc.barrier()

# Step 2: Random Oracle
qc.append(oracle,[0,1,2,3])
qc.barrier()

# Step 3: 다시 Hadamard
qc.h([0,1,2])
qc.barrier()

# Step 4: 측정
qc.measure([0, 1, 2], [0, 1, 2])
qc.draw("mpl")

In [ ]:
q_simulation(qc,1)

# Problem 2

Bob은 입력이 3개($n=3$)인 함수 $f(x)$ 의 성질을 판별하려 합니다. 이 함수 역시 **constant** 이거나 **balanced** 둘 중 하나입니다.

고전 컴퓨터라면 worst case에 함수를 **5번** 호출해야 하지만, Bob은 양자 컴퓨터로 **단 한 번의 실행** 만에 판별하려 합니다.

**(2a) 항상 1을 반환하는 Constant Oracle 만들기**
- 4개의 큐비트를 사용하는 `Random Oracle`이라는 이름의 회로를 만들고, 입력과 무관하게 항상 1을 출력하는 함수를 구현하세요.
- 이 오라클이 잘 작동하는지 확인하기 위해, 4 큐비트 + 고전 비트 1개를 가진 시각화용 회로 `oracle_for_visual`을 만드세요.
- 입력 큐비트에 Hadamard를 건 뒤 오라클을 통과시키고 보조 큐비트를 측정했을 때, 결과가 **항상 1** 인지 `q_simulation`으로 확인하세요.

**(2b) Deutsch–Jozsa로 constant 판별하기**
- (2a)에서 만든 오라클을 `qc.append()`로 회로에 추가하세요.
- `q_simulation(qc, 1)`의 결과가 Deutsch–Jozsa 이론과 일치해야 합니다. (constant이므로 측정 결과는 **000**)

In [ ]:
# (2a): 코드를 완성하시오.
oracle = QuantumCircuit(4,name='Random Oracle')

oracle.draw("mpl")

In [ ]:
# (2a): 코드를 완성하시오.
oracle_for_visual = QuantumCircuit(4,1)


oracle_for_visual.barrier() # Oracle Starts

oracle_for_visual.barrier()

oracle_for_visual.draw("mpl")

In [ ]:
# Checking Constant / Balance
q_simulation(oracle_for_visual,1000)

In [ ]:
#2b: 코드를 완성하시오.
qc = QuantumCircuit(4, 3)

# Step 1: 초기화
       # 결과 큐비트 |1⟩
qc.barrier()
      # Hadamard 변환
qc.barrier()

# Step 2: Random Oracle

qc.barrier()

# Step 3: 다시 Hadamard

qc.barrier()

# Step 4: 측정


qc.draw("mpl")

In [ ]:
q_simulation(qc, 1)

# 3. Bernstein–Vazirani Algorithm

이번엔 오라클이 **비밀 비트 문자열** $s$ 를 숨기고 있습니다. 함수는 입력과 비밀을 다음처럼 섞습니다.

$$
f(x) = (s_0\,x_0) \oplus (s_1\,x_1) \oplus \cdots \pmod 2
$$

**할 일:** 비밀 $s$ 를 알아내기.

고전 컴퓨터는 query 한 번에 **$s$ 의 비트 하나** 밖에 못 얻으므로, 전체를 알려면 **$n$ 번** 물어야 합니다. Bernstein–Vazirani는 Deutsch–Jozsa와 똑같은 회로로, 비밀 전체를 단 **1번의 query** 에 찾아냅니다.

오라클 만드는 법: 비밀에서 1인 자리마다 보조 큐비트로 향하는 CNOT을 하나씩 추가합니다. 알고리즘이 끝나면 입력 큐비트가 **정확히 비밀 문자열** 로 모이므로, 그냥 측정하면 $s$ 가 바로 읽힙니다.

In [ ]:
def bv_oracle(s: str):
    # s의 길이에 보조 큐비트 1개를 더한 크기의 회로 생성
    n = len(s)
    qc = QuantumCircuit(n + 1)
    for index, bit in enumerate(s):
        if bit == "1":
            qc.cx(index, n) # index번째 비트와 보조 큐비트(n) 연결

    qc.name = f"BV_Oracle_{s}"
    return qc

# 오라클 구조 확인 (예: "1011")
s_target = "1011"
display(bv_oracle(s_target).draw(output="mpl"))

# Problem 3

Alice는 비밀 비트 문자열 $s = \texttt{1011}$ 을 숨기고 있는 오라클을 가지고 있습니다. 이 오라클은 입력 $x$ 에 대해 $s \cdot x \bmod 2$ 를 계산해 돌려줍니다 (위에서 정의한 `bv_oracle`이 이를 구현합니다).

Bob은 Bernstein–Vazirani 알고리즘으로 **단 한 번의 실행** 만에 비밀 문자열 $s$ 를 알아내려 합니다.

**요구사항**
- 단 한 번의 실행으로 $s$ 를 알아내는 회로를 완성하세요.
- `q_simulation(qc, 1)`의 결과가 `1011`이 나와야 합니다.

**HINT:** Bernstein–Vazirani는 Deutsch–Jozsa와 **같은 회로 구조** 를 사용합니다. 차이는 측정 결과를 읽는 의미뿐입니다. Deutsch–Jozsa에선 "전부 0인가 아닌가"만 봤지만, 여기선 측정된 비트 문자열 자체가 곧 비밀 $s$ 입니다.

In [ ]:
# 코드를 완성하시오.
n = len(s_target)
bv_circuit = QuantumCircuit(n + 1, n)

# Step 1: Initialize qubits
# 보조 큐비트을 |-> 상태로 만듦


# Step 2: Apply the Oracle
bv_circuit.append(bv_oracle(s_target), range(n + 1))
bv_circuit.barrier()

# Step 3: Apply H-gates


# Step 4: Measurement
bv_circuit.measure(range(n), range(n))

# 회로 그리기
display(bv_circuit.draw(output="mpl"))

In [ ]:
q_simulation(bv_circuit, 1)

# 4. Simon's Algorithm

Simon의 알고리즘은 지금까지 중 양자와 고전의 차이가 **가장 크게** 벌어지는 예이고, 그 유명한 Shor의 소인수분해 알고리즘에 영감을 준 알고리즘입니다.

오라클은 비밀 문자열 $s$ 를 숨기고 있습니다. 함수는 **2-to-1** 입니다. 즉 입력들이 같은 출력을 내는 짝으로 묶입니다. 두 입력 $x$ 와 $y$ 가 같은 출력을 내는 건 정확히 다음일 때입니다.

$$
y = x \oplus s
$$

**할 일:** 비밀 주기 $s$ 를 찾기.

고전 컴퓨터는 같은 출력을 내는 짝을 우연히 마주칠 때까지 계속 물어봐야 하는데, 이게 **엄청나게 많은(지수적인)** 횟수가 될 수 있습니다. Simon의 알고리즘은 약 **$n$ 번의 실행** 이면 됩니다.

핵심은 이것입니다. 회로를 돌릴 때마다 측정 결과 $z$ 는 비밀과 다음 간단한 규칙을 만족합니다.

$$
z\cdot s = 0 \pmod 2
$$

(이는 $z$ 와 $s$ 가 mod 2 계산에서 "수직"이라는 뜻입니다.) 이런 결과를 몇 개 모으면, 간단한 일반 계산만으로 $s$ 를 복원할 수 있습니다. 어려운 부분은 양자 컴퓨터가 하고, 마무리는 쉬운 고전 계산이 해줍니다.

### Simon 오라클 정의하기

오라클은 $2n$ 개의 큐비트(입력 $n$ 개 + 출력 $n$ 개) 위에서 두 단계로 만들어집니다.

1. **복사:** 각 큐비트에 CNOT을 걸어 입력을 출력 쪽으로 복사합니다. $|x\rangle|0\rangle$ 이 $|x\rangle|x\rangle$ 이 됩니다.
2. **주기 더하기:** $s$ 의 첫 번째 `1`을 제어 큐비트로 삼아, 추가 CNOT들로 2-to-1 짝짓기 구조를 만듭니다.

만약 $s = 0000$ 이면 두 번째 단계가 생략되고 함수는 그냥 일대일(1-to-1)이 됩니다. 예시는 $s = \texttt{1001}$ 을 사용합니다.

In [ ]:
def simon_oracle(b):
 """returns a Simon oracle for bitstring b"""
 #b = b[::-1] # reverse b for easy iteration
 n = len(b)
 qc = QuantumCircuit(n*2)
 # Do copy; |x>|0> -> |x>|x>
 for q in range(n):
     qc.cx(q, q+n)
 if '1' not in b:
     return qc # 1:1 mapping, so just exit
 i = b.find('1') # index of first non-zero bit in b
 # Do |x> -> |s.x> on condition that q_i is 1
 for q in range(n):
     if b[q] == '1':
         qc.cx(i, (q)+n)
 return qc

In [ ]:
s="1001"
simon_oracle(s).draw(output="mpl")

### Simon 알고리즘 구현하기

이제 $2n$ 큐비트(입력 $n$ 개 + 출력 $n$ 개) 위에 전체 회로를 조립합니다.

1. 입력 큐비트 $n$ 개에 Hadamard를 걸어 모든 입력을 동시에 올립니다.
2. Simon 오라클을 한 번 통과시킵니다.
3. 입력 큐비트 $n$ 개에 다시 Hadamard를 겁니다.
4. 입력 큐비트를 측정합니다.

`q_simulation(qc, 1000)`처럼 여러 번 돌려서 측정 결과 $z$ 들을 모읍니다. 이렇게 모은 $z$ 들이 비밀 $s$ 와 어떤 관계인지는 이후 해석(Interpretation) 부분에서 확인합니다.

# Problem 4

Bob이 가진 오라클은 비밀 문자열 $s = \texttt{1001}$ 을 숨기고 있습니다 (위 `simon_oracle`로 구현). 측정 결과 $z$ 로부터 $z \cdot s \bmod 2 = 0$ 이라는 단서를 모아 $s$ 를 찾는 것이 목표입니다.

**요구사항**
- 제공된 `simon_oracle(s)`로 $s = \texttt{1001}$ 을 인코딩하는 8 큐비트(입력 4 + 출력 4) 회로를 만들고, 입력·출력 큐비트가 어떻게 연결되는지 확인하세요.
- 이 오라클로 Simon 회로를 완성하고 실행하세요.
- `bdotz(s, z)`로 측정 결과들이 **모두** $s = \texttt{1001}$ 과 직교함($z\cdot s \bmod 2 = 0$)을 확인하세요.

In [ ]:
# 코드를 완성하시오.
qc = QuantumCircuit(8, 4)

qc.draw("mpl")

In [ ]:
q_simulation(qc,1000)

### 해석: 내적(mod 2)은 항상 0일까?

`### Implementing`에서 모은 측정 결과 $z$ 는 무작위가 아닙니다. Simon 알고리즘은 **모든** $z$ 가 비밀 $s$ 와 다음 관계를 만족하도록 보장합니다.

$$
z\cdot s = (z_0 s_0) + (z_1 s_1) + \cdots \pmod 2 = 0
$$

(이는 $z$ 와 $s$ 가 mod 2 계산에서 "수직"이라는 뜻입니다.) 이렇게 얻은 식들을 여러 개 모으면 거꾸로 $s$ 를 알아낼 수 있고, 그게 Simon 알고리즘이 $s$ 를 효율적으로 찾는 원리입니다.

아래 코드는 $s = \texttt{1001}$ 을 써서 측정된 각 $z$ 에 대해 이 값을 계산합니다. 모든 결과가 **0** 으로 나오면, 오라클이 제대로 작동하고 우리가 $s$ 를 찾는 단서를 정확히 얻고 있다는 뜻입니다.

In [ ]:
counts = q_simulation(qc,1000,return_count=True)
counts = list(counts.keys())
counts

In [ ]:
# 내적 계산 함수

def bdotz(s, z):
    accum = 0
    for i in range(len(s)):
        accum += int(s[i]) * int(z[i])
    return (accum % 2)

In [ ]:
for z in counts:
    print( '{}.{} = {} (mod 2)'.format(s, z, bdotz(s,z)) )